# 00 — Setup

**Just run this notebook top to bottom.** Each cell is idempotent — safe to
re-run any time. When it finishes, every other notebook will import and run.

## What it does

1. **Verifies Python ≥ 3.11** (arc requires it).
2. **Clones arc** into `vendor/arc/` (skipped if already present).
3. **Installs `requirements.txt`** into the kernel you're running — the arc packages, jupyter, spaCy, matplotlib, tiktoken.
4. **Loads the spaCy NER model** from the vendored copy in `data/models/`.
5. **Reads `.env`** to find your active provider (`ARC_PROVIDER`).
6. **Verifies imports** — every package the curriculum uses.
7. **Runs a smoke-test LLM call** through that provider.

The curriculum is notebooks **01–10** (`01_context_engineering` → `10_your_workflow`).

**One switch for every machine.** The notebooks never change between your laptop,
a hosted lab, or a local GPU box — only `.env` does:

| Where | `.env` |
|---|---|
| Anthropic cloud (laptop / hosted lab) | `ARC_PROVIDER=anthropic` + `ANTHROPIC_API_KEY` |
| Local LiteLLM proxy | `ARC_PROVIDER=litellm` + `ARC_BASE_URL` + `ARC_MODEL` |
| Air-gapped Ollama | `ARC_PROVIDER=ollama` + `ARC_BASE_URL` + `ARC_MODEL` |

*(Local dev shortcut: running `./setup.sh` in a terminal does steps 1–6 and
registers an "Arc venv" kernel — but on a hosted lab you don't need it.)*

## 1 — Python version

Arc and the curriculum require Python 3.11 or newer. If this fails, install a newer Python (`brew install python@3.12` on macOS, or use `pyenv`) and restart the kernel.

In [1]:
import sys

version = sys.version_info
assert version >= (3, 11), (
    f"Python {version.major}.{version.minor} found; arc requires 3.11+. "
    f"Install a newer Python and select its kernel."
)
print(f"✓ Python {version.major}.{version.minor}.{version.micro}")

✓ Python 3.12.13


## 2 — Vendor arc

The five `-e vendor/arc/packages/...` lines in `requirements.txt` need [Arc](https://github.com/joshuamschultz/Arc) checked out at `vendor/arc/`. This cell clones it for you.

Idempotent — if `vendor/arc/` already exists, it skips. To upgrade arc later, delete `vendor/arc/` and re-run.

**Air-gapped labs**: pre-stage Arc at `vendor/arc/` before opening this notebook; the cell will detect it and skip the network call.

In [2]:
import subprocess
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
arc_dir = project_root / "vendor" / "arc"
arc_url = "https://github.com/joshuamschultz/Arc.git"

if arc_dir.exists():
    print(f"✓ arc already vendored at {arc_dir.relative_to(project_root)}")
else:
    arc_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {arc_url} → {arc_dir.relative_to(project_root)} …")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", arc_url, str(arc_dir)],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("git clone failed — see stderr above")
    print(f"✓ Cloned arc to {arc_dir.relative_to(project_root)}")

required = ["arctrust", "arcllm", "arcrun", "arcagent", "arcskill"]
missing = [p for p in required if not (arc_dir / "packages" / p).exists()]
if missing:
    raise RuntimeError(f"arc clone is missing packages: {missing}")
print(f"  Packages present: {', '.join(required)}")


✓ arc already vendored at vendor/arc
  Packages present: arctrust, arcllm, arcrun, arcagent, arcskill


## 3 — Install dependencies

Installs everything in `requirements.txt` **into the kernel you're running right now**:

- **Notebook runtime**: `jupyter`, `ipykernel`, `ipywidgets`
- **Charts + tokens**: `matplotlib`, `numpy`, `tiktoken`
- **Config + display**: `python-dotenv`, `rich`
- **NER for PII redaction (NB08)**: `spacy`
- **Arc packages** (editable, from `vendor/arc/packages/`): `arctrust`, `arcllm`, `arcrun`, `arc-agent`, `arcskill`

On a hosted lab this is your own container, so installing here is exactly right —
no terminal, no separate virtualenv. (Locally, if you ran `./setup.sh`, this
installs into that `.venv` instead.)

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
req_file = project_root / "requirements.txt"
assert req_file.exists(), f"requirements.txt not found at {req_file}"

# Install into whichever interpreter is actually running this notebook:
#   • Local dev   → the "Arc venv" kernel (the project .venv from setup.sh)
#   • Hosted lab  → the JupyterHub kernel in your container (no .venv, and
#                   that's fine — it's your own environment to install into).
# Either way "where I'm running" is the right target, so attendees never need
# a terminal or a separate venv.
venv_py = project_root / ".venv" / "bin" / "python"
target = str(venv_py) if venv_py.exists() else sys.executable

uv = shutil.which("uv")
if uv:
    cmd = [uv, "pip", "install", "--python", target, "-r", str(req_file)]
else:
    cmd = [target, "-m", "pip", "install", "-r", str(req_file)]

# Some hub images set PIP_REQUIRE_VIRTUALENV=true to block base-env installs;
# this is a single-user container, so installing here is intended.
env = {**os.environ, "PIP_REQUIRE_VIRTUALENV": "false"}

print(f"Installing requirements into:\n  {target}\n(this can take a minute the first time)…")
result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True, env=env)
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr[-2000:])
    raise RuntimeError("pip install failed — see stderr above.")
print(f"✓ Dependencies installed into {target}")

## 4 — spaCy NER model

NB08 §2b uses spaCy for PII detection (names, orgs, locations). The model is **vendored** at `data/models/en_core_web_sm-3.8.0/` so air-gapped labs work with zero downloads.

This cell:
1. Tries to load the vendored model from disk.
2. Falls back to the pip-registered model name.
3. If neither works, suggests `python -m spacy download en_core_web_sm`.

In [4]:
import spacy
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
vendored = project_root / "data" / "models" / "en_core_web_sm-3.8.0"

nlp = None
if vendored.exists():
    nlp = spacy.load(str(vendored))
    print(f"✓ Loaded vendored model · {vendored.relative_to(project_root)}")
else:
    try:
        nlp = spacy.load("en_core_web_sm")
        print("✓ Loaded pip-installed en_core_web_sm")
    except OSError:
        print("⚠ No spaCy model found.")
        print("   Run:  python -m spacy download en_core_web_sm")
        print("   (NB08 will not work until this is resolved.)")

if nlp is not None:
    doc = nlp("Sarah Chen reviewed the report at Sandia National Labs.")
    ents = [(e.text, e.label_) for e in doc.ents]
    print(f"  Smoke test: {ents}")

✓ Loaded vendored model · data/models/en_core_web_sm-3.8.0
  Smoke test: [('Sarah Chen', 'PERSON'), ('Sandia National Labs', 'ORG')]


## 5 — Environment file (`.env`)

Notebooks read the **active provider** from `.env`. If it doesn't exist, this cell
copies `.env.example` and prompts you to pick one block.

Set **one switch** — `ARC_PROVIDER` — plus its credential:
- `ARC_PROVIDER=anthropic` + `ANTHROPIC_API_KEY` — cloud (laptop / hosted lab)
- `ARC_PROVIDER=litellm` + `ARC_BASE_URL` (+ `ARC_MODEL`) — local LiteLLM proxy
- `ARC_PROVIDER=ollama` + `ARC_BASE_URL` (+ `ARC_MODEL`) — air-gapped

Every notebook calls `setup()` / `connect()`, which read these — so nothing else
changes between machines.

In [ ]:
import shutil
import os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
env_file = project_root / ".env"
env_example = project_root / ".env.example"

if not env_file.exists():
    shutil.copy(env_example, env_file)
    print(f"✓ Created {env_file.relative_to(project_root)} from .env.example")
    print("  → Open it, pick ONE provider block, then re-run this cell.")
else:
    print(f"✓ {env_file.relative_to(project_root)} already exists")

from dotenv import load_dotenv

load_dotenv(env_file)

# One switch decides everything: ARC_PROVIDER (+ ARC_MODEL / ARC_BASE_URL).
provider = (os.getenv("ARC_PROVIDER") or "anthropic").lower()
print(f"\nActive provider (ARC_PROVIDER): {provider}")

if provider == "anthropic":
    ready = bool(os.getenv("ANTHROPIC_API_KEY"))
    print(f"  {'✓' if ready else '·'} ANTHROPIC_API_KEY")
elif provider in ("openai", "litellm"):
    if provider == "litellm":
        print(f"  base_url: {os.getenv('ARC_BASE_URL', 'http://localhost:4000')}")
    has_key = bool(os.getenv("OPENAI_API_KEY"))
    print(f"  {'✓' if has_key else '·'} OPENAI_API_KEY (local proxies may not need one)")
    ready = True
elif provider == "ollama":
    print(f"  base_url: {os.getenv('ARC_BASE_URL', 'http://localhost:11434')}")
    ready = True
else:
    print(f"  ⚠ unknown provider '{provider}' — expected anthropic | litellm | ollama")
    ready = False

if not ready:
    print("\n⚠ Provider not ready. Edit .env, then re-run this cell.")

## 6 — Verify imports

Imports every package the curriculum touches. If any line fails, re-run §3 — pip didn't finish.

In [6]:
import importlib

modules = [
    # arc
    "arctrust",
    "arcllm",
    "arcrun",
    "arcagent",
    "arcskill",
    # third-party
    "spacy",
    "dotenv",
    "rich",
    "ipywidgets",
    "pydantic",
    "httpx",
    "matplotlib",
    "numpy",
    "tiktoken",
]

missing = []
for name in modules:
    try:
        mod = importlib.import_module(name)
        version = getattr(mod, "__version__", "?")
        print(f"  ✓ {name:<12} {version}")
    except ImportError as e:
        print(f"  ✗ {name:<12} — {e}")
        missing.append(name)

if missing:
    raise RuntimeError(f"Missing modules: {missing}. Re-run §3 (or ./setup.sh).")
print("\n✓ All imports succeeded")

  ✓ arctrust     0.2.0
  ✓ arcllm       0.4.0
  ✓ arcrun       0.5.0
  ✓ arcagent     0.4.0
  ✓ arcskill     0.1.0
  ✓ spacy        3.8.14
  ✓ dotenv       ?
  ✓ rich         ?
  ✓ ipywidgets   8.1.8
  ✓ pydantic     2.13.4
  ✓ httpx        0.28.1


  ✓ matplotlib   3.10.9
  ✓ numpy        2.4.6
  ✓ tiktoken     0.13.0

✓ All imports succeeded


## 7 — Smoke-test LLM call

Sends one prompt through arc to confirm the harness, network, and credentials all
work end-to-end. It uses the same `connect()` wiring as every notebook, so it
tests **whatever `ARC_PROVIDER` you set in §5** — no edits needed.

In [ ]:
import os
from roadshow import connect, Message

# Uses the same one-switch wiring as every notebook: connect() reads
# ARC_PROVIDER / ARC_MODEL / ARC_BASE_URL from .env. No per-machine edits.
try:
    model = connect()
    resp = await model.invoke([Message(role="user", content="In one sentence: what is a harness?")])
    print(f"Provider: {os.getenv('ARC_PROVIDER', 'anthropic')}")
    print(f"Model:    {resp.model}")
    print(f"Reply:    {resp.content}")
    print("\n✓ End-to-end LLM call works")
except Exception as exc:
    print(f"⊘ Smoke test skipped: {type(exc).__name__}: {exc}")
    print("  Set ARC_PROVIDER in .env (+ the matching key or ARC_BASE_URL), then re-run.")

## You're set up

Open **`01_context_engineering.ipynb`** and start the curriculum (01 → 10).

If something broke, re-run the failing section — each step is idempotent and
safe to retry.